# Hafta 15 · QML IV: Uçtan Uca Proje ve Klasik–Kuantum Karşılaştırma
**Ders:** Kuantum Hesaplama ve Uygulamaları · **Lab süresi:** ~60 dk · **Ortam:** Google Colab (CPU yeterli)

Bu son lab'da 12–14. haftalarda öğrendiğimiz her şeyi **tek bir proje boru hattında (pipeline)** birleştiriyoruz: veri → ön işleme → klasik temel modeller → QSVM → VQC → değerlendirme (doğruluk, F1, karışıklık matrisi, 5-kat CV, güven aralığı) → maliyet → gürültü → rapor tablosu. Bu notebook, dönem projeniz için **başlangıç şablonu** olarak da kullanılabilir.

| Bölüm | Konu | Süre |
|---|---|---|
| 0 | Kurulum, veri setleri, yardımcılar | 5 dk |
| A | Veri ve ön işleme: PCA, açıklanan varyans, stratify ayrım | 7 dk |
| B | Klasik temel modeller: lojistik regresyon, RBF-SVM, küçük MLP | 5 dk |
| C | QSVM: Statevector çekirdeği (13. hafta yöntemi) + bant genişliği seçimi | 10 dk |
| D | VQC: PennyLane + PyTorch hibrit model (14. hafta) | 10 dk |
| E | Değerlendirme: F1, karışıklık matrisi, 5-kat CV, güven aralığı | 8 dk |
| F | Maliyet analizi: devre sayısı × shot, simülasyon süresi | 5 dk |
| G | Gürültü deneyi: Aer gürültü modeliyle QSVM çekirdeği | 7 dk |
| H | Rapor tablosu: pandas → CSV / markdown | 3 dk |
| I | Alıştırmalar | ödev |

## 0 · Kurulum
`torch` Colab'da hazır bulunur, ayrıca kurmayın.

In [ ]:
!pip install -q qiskit qiskit-aer pylatexenc pennylane scikit-learn

In [ ]:
import time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from scipy import stats
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import zz_feature_map
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
NAVY, BLUE, ORANGE, GRAY = "#1F3A5F", "#2E6DB4", "#D9822B", "#8A94A6"
SEED = 42
T_START = time.time()
print("hazır")

### Veri setleri
Aşağıdaki fonksiyonlar ders boyunca kullandığımız `datasets.py` dosyasından **aynen** kopyalanmıştır. Veriler sklearn içindeki `digits` setinden üretilir; **internet gerekmez**. Tüm özellikler açı kodlaması için [0, π] aralığına ölçeklenmiştir.

Bu veri setlerinin CSV'leri ayrıca verilmiştir: `digits01_pca4.csv`, `digits01_pca8.csv`, `digits38_pca6.csv`.

In [ ]:
def ds_digits01(n_components=4):
    """sklearn digits (8x8, internet gerekmez): 0 ve 1 rakamları, PCA ile n_components özelliğe indirgenmiş."""
    d = load_digits()
    m = d.target < 2
    Xp = PCA(n_components=n_components, random_state=SEED).fit_transform(d.data[m])
    X = MinMaxScaler((0, np.pi)).fit_transform(Xp)
    df = pd.DataFrame(X, columns=[f"pc{i+1}" for i in range(n_components)]); df["y"] = d.target[m]
    return df

def ds_digits_pair(a=3, b=8, n_components=6):
    """Daha zor ikili problem (3 vs 8) — 15. hafta projesi için."""
    d = load_digits()
    m = (d.target == a) | (d.target == b)
    Xp = PCA(n_components=n_components, random_state=SEED).fit_transform(d.data[m])
    X = MinMaxScaler((0, np.pi)).fit_transform(Xp)
    df = pd.DataFrame(X, columns=[f"pc{i+1}" for i in range(n_components)]); df["y"] = (d.target[m] == b).astype(int)
    return df

CASES = {"0–1 (PCA 4)": ds_digits01(4), "0–1 (PCA 8)": ds_digits01(8), "3–8 (PCA 6)": ds_digits_pair(3, 8, 6)}
for k, df in CASES.items():
    print(f"{k:12s} şekil = {df.shape}   sınıf dağılımı = {df.y.value_counts().to_dict()}")

---
## A · Veri ve ön işleme
**Boru hattı adımları:** 64 piksel → PCA ile k bileşen (k = kübit sayısı) → MinMax ile [0, π] → %70 eğitim / %30 test, **stratify** ile sınıf oranları korunarak.

⚠️ **Veri sızıntısı (data leakage) uyarısı:** `datasets.py` PCA ve ölçeklemeyi tüm veri üzerinde yapar (ders içi kolaylık için). Gerçek bir projede PCA ve ölçekleyici **yalnızca eğitim kümesinde** `fit` edilmeli, test kümesine sadece `transform` uygulanmalıdır. Bölüm A'nın sonunda doğru sürümü görüyoruz.

In [ ]:
d = load_digits()
fig, axs = plt.subplots(2, 8, figsize=(11, 3))
for r_, digs in enumerate([(0, 1), (3, 8)]):
    for c in range(8):
        dg = digs[c // 4]; idx = np.where(d.target == dg)[0][c % 4]
        axs[r_, c].imshow(d.images[idx], cmap="Greys"); axs[r_, c].axis("off"); axs[r_, c].set_title(f"'{dg}'", fontsize=9)
plt.suptitle("Üst: Vaka 1 (0 vs 1, kolay)   Alt: Vaka 2 (3 vs 8, zor)", color=NAVY); plt.tight_layout(); plt.show()

# PCA açıklanan varyans
for (a, b) in [(0, 1), (3, 8)]:
    m = (d.target == a) | (d.target == b)
    ev = PCA(10, random_state=SEED).fit(d.data[m]).explained_variance_ratio_
    print(f"{a} vs {b}: ilk 4 bileşen %{100*ev[:4].sum():.1f}, ilk 6 %{100*ev[:6].sum():.1f}, ilk 8 %{100*ev[:8].sum():.1f}")

In [ ]:
def split(df, test_size=0.3):
    X = df.drop(columns="y").values; y = df.y.values
    return train_test_split(X, y, test_size=test_size, stratify=y, random_state=SEED)

SPLITS = {k: split(df) for k, df in CASES.items()}
for k, (Xtr, Xte, ytr, yte) in SPLITS.items():
    print(f"{k:12s} eğitim {Xtr.shape}  test {Xte.shape}  test'te sınıf 1 oranı = {yte.mean():.2f}")

# Sızıntısız (doğru) ön işleme: PCA + ölçekleyici sadece eğitimde fit edilir
def leak_free(a=3, b=8, k=6):
    m = (d.target == a) | (d.target == b); X = d.data[m]; y = (d.target[m] == b).astype(int)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEED)
    pca = PCA(k, random_state=SEED).fit(Xtr); sc = MinMaxScaler((0, np.pi)).fit(pca.transform(Xtr))
    return sc.transform(pca.transform(Xtr)), np.clip(sc.transform(pca.transform(Xte)), 0, np.pi), ytr, yte
Xtr_lf, Xte_lf, ytr_lf, yte_lf = leak_free()
print("Sızıntısız 3–8, RBF-SVM test doğruluğu:", SVC().fit(Xtr_lf, ytr_lf).score(Xte_lf, yte_lf))

---
## B · Klasik temel modeller (baseline)
Bir kuantum modelin iyi olup olmadığını söyleyebilmek için önce **güçlü ve ayarlanmış klasik modellerle** karşılaştırmalıyız. Burada üç tanesini kullanıyoruz: lojistik regresyon (doğrusal), RBF çekirdekli SVM (doğrusal olmayan) ve 16 nöronlu küçük bir MLP.

In [ ]:
def classical_models():
    return {"Lojistik R.": LogisticRegression(max_iter=1000),
            "RBF-SVM": SVC(kernel="rbf"),
            "MLP (16)": MLPClassifier((16,), max_iter=2000, random_state=SEED)}

RESULTS = []        # rapor tablosu için her satır: vaka, model, doğruluk, F1, süre
PREDS = {}          # karışıklık matrisleri için tahminler
def record(case, model, yte, pred, sec):
    RESULTS.append({"vaka": case, "model": model, "dogruluk": accuracy_score(yte, pred), "F1": f1_score(yte, pred), "sure_s": sec})
    PREDS[(case, model)] = pred

for case, (Xtr, Xte, ytr, yte) in SPLITS.items():
    for name, m in classical_models().items():
        t = time.time(); m.fit(Xtr, ytr); p = m.predict(Xte)
        record(case, name, yte, p, time.time() - t)
pd.DataFrame(RESULTS).pivot(index="model", columns="vaka", values="dogruluk").round(3)

---
## C · QSVM: kuantum çekirdek (13. hafta yöntemi)
Özellik haritası olarak `zz_feature_map` (reps = 2, doğrusal dolanıklık) kullanıyoruz. Çekirdek değeri, iki özellik durumunun örtüşmesidir:
$$K(x, x') = |\langle \phi(x) | \phi(x') \rangle|^2$$
Simülatörde her örnek için **bir kez** durum vektörünü hesaplayıp tüm çekirdeği tek matris çarpımıyla alıyoruz: `K = |S_A^* S_B^T|^2`.

**Bant genişliği (bandwidth):** Özellikleri bir `bw` çarpanıyla küçültmek (x → bw·x), çekirdek değerlerinin sıfıra yığılmasını (yoğunlaşma) önler. `bw` bir **hiperparametredir** ve yalnızca eğitim kümesinde CV ile seçilir.

In [ ]:
fm = zz_feature_map(4, reps=1, entanglement="linear")
display(fm.draw("mpl", fold=-1))

# Çekirdek devresi: U(x) ardından U(x')†, sonra tüm kübitler 0 okunma olasılığı = K(x, x')
xa, xb = ParameterVector("x", 2), ParameterVector("x'", 2)
f2 = zz_feature_map(2, reps=1)
kc = f2.assign_parameters(xa).compose(f2.assign_parameters(xb).inverse()); kc.measure_all()
display(kc.draw("mpl", fold=-1))

In [ ]:
def fm_states(X, bw=0.25, reps=2):
    fm = zz_feature_map(X.shape[1], reps=reps, entanglement="linear")
    return np.array([Statevector(fm.assign_parameters(x * bw)).data for x in X])

def kmat(SA, SB):
    return np.abs(SA.conj() @ SB.T) ** 2

def cv_precomputed(K, y, k=5):
    skf = StratifiedKFold(k, shuffle=True, random_state=SEED); sc = []
    for a, b in skf.split(K, y):
        sc.append(SVC(kernel="precomputed").fit(K[np.ix_(a, a)], y[a]).score(K[np.ix_(b, a)], y[b]))
    return np.array(sc)

# Bant genişliği seçimi: SADECE eğitim kümesi üzerinde 5-kat CV
Xtr, Xte, ytr, yte = SPLITS["3–8 (PCA 6)"]
for bw in [0.1, 0.25, 0.5, 1.0]:
    S = fm_states(Xtr, bw); K = kmat(S, S)
    print(f"bw = {bw:<5} CV doğruluğu = {cv_precomputed(K, ytr).mean():.3f}   köşegen dışı ort. K = {K[~np.eye(len(K), dtype=bool)].mean():.3f}")

In [ ]:
BW = 0.25   # derste seçilen değer (üç vakada da iyi çalışıyor)
KERNELS = {}
for case, (Xtr, Xte, ytr, yte) in SPLITS.items():
    t = time.time(); Str, Ste = fm_states(Xtr, BW), fm_states(Xte, BW)
    Ktr, Kte = kmat(Str, Str), kmat(Ste, Str); KERNELS[case] = (Ktr, Kte)
    p = SVC(kernel="precomputed").fit(Ktr, ytr).predict(Kte)
    record(case, "QSVM", yte, p, time.time() - t)
    print(f"{case:12s} QSVM test doğruluğu = {accuracy_score(yte, p):.3f}   ({time.time()-t:.1f} s)")

Ktr = KERNELS["3–8 (PCA 6)"][0]; o = np.argsort(SPLITS["3–8 (PCA 6)"][2])
plt.imshow(Ktr[np.ix_(o, o)], cmap="Blues", vmin=0, vmax=1); plt.colorbar()
plt.title("3–8 eğitim çekirdeği (önce 3'ler, sonra 8'ler)", color=NAVY); plt.axis("off"); plt.show()

---
## D · VQC: PennyLane + PyTorch hibrit model (14. hafta)
Model: açı kodlaması (`AngleEmbedding`, RY) → 2 katman `StronglyEntanglingLayers` → ⟨Z₀⟩. Çıkış `3·(⟨Z₀⟩ + b)` bir logit olarak ikili çapraz entropiye verilir. Optimizasyon: Adam, lr = 0.1, 40 epoch, tam yığın (full batch).

In [ ]:
import torch, pennylane as qml

def train_vqc(Xtr, ytr, Xte, layers=2, epochs=40, lr=0.1):
    n = Xtr.shape[1]; dev = qml.device("default.qubit", wires=n)
    @qml.qnode(dev, interface="torch")
    def circ(x, w):
        qml.AngleEmbedding(x, wires=range(n), rotation="Y")
        qml.StronglyEntanglingLayers(w, wires=range(n))
        return qml.expval(qml.PauliZ(0))
    torch.manual_seed(0)
    w = torch.nn.Parameter(0.1 * torch.randn(layers, n, 3, dtype=torch.float64))
    b = torch.nn.Parameter(torch.zeros(1, dtype=torch.float64))
    opt = torch.optim.Adam([w, b], lr=lr)
    Xt, yt = torch.tensor(Xtr), torch.tensor(ytr, dtype=torch.float64); losses = []
    for ep in range(epochs):
        opt.zero_grad()
        loss = torch.nn.functional.binary_cross_entropy_with_logits(3 * (circ(Xt, w) + b), yt)
        loss.backward(); opt.step(); losses.append(loss.item())
    with torch.no_grad():
        pred = ((circ(torch.tensor(Xte), w) + b) > 0).numpy().astype(int)
    return pred, losses, circ, w

LOSSES = {}
for case, (Xtr, Xte, ytr, yte) in SPLITS.items():
    t = time.time(); p, losses, circ, w = train_vqc(Xtr, ytr, Xte); LOSSES[case] = losses
    record(case, "VQC", yte, p, time.time() - t)
    print(f"{case:12s} VQC test doğruluğu = {accuracy_score(yte, p):.3f}   ({time.time()-t:.1f} s, {w.numel()+1} parametre)")

for case, l in LOSSES.items(): plt.plot(l, label=case)
plt.xlabel("epoch"); plt.ylabel("kayıp"); plt.legend(); plt.title("VQC eğitim kaybı", color=NAVY); plt.show()

In [ ]:
# Son eğitilen (3–8) devrenin çizimi (6 kübit, 2 katman)
fig, ax = qml.draw_mpl(circ, decimals=1)(torch.tensor(SPLITS["3–8 (PCA 6)"][1][0]), w.detach())
plt.show()

---
## E · Değerlendirme: F1, karışıklık matrisi, 5-kat CV, güven aralığı
- **Karışıklık matrisi** `[[TN, FP], [FN, TP]]` (sklearn düzeni; satır = gerçek, sütun = tahmin; pozitif sınıf = 1)
- **Kesinlik** P = TP / (TP + FP), **Duyarlılık** R = TP / (TP + FN), **F1** = 2PR / (P + R)
- **5-kat CV:** tek bir ayrım şansa bağlıdır; 5 farklı ayrımın ortalaması ve standart sapması daha güvenilirdir.
- **%95 güven aralığı (t dağılımı, k = 5):** ort ± t₀.₉₇₅,₄ · s / √5, t₀.₉₇₅,₄ ≈ 2.776

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, m in zip(axs, ["Lojistik R.", "RBF-SVM", "QSVM", "VQC"]):
    yte = SPLITS["3–8 (PCA 6)"][3]; cm = confusion_matrix(yte, PREDS[("3–8 (PCA 6)", m)])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2): ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=14, color="white" if cm[i, j] > cm.max()/2 else "black")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.set_xticklabels(["3", "8"]); ax.set_yticklabels(["3", "8"])
    ax.set_xlabel("tahmin"); ax.set_ylabel("gerçek"); ax.set_title(m, color=NAVY)
plt.tight_layout(); plt.show()

In [ ]:
def cv_scores(case, model):
    df = CASES[case]; X = df.drop(columns="y").values; y = df.y.values
    if model == "QSVM":
        S = fm_states(X, BW); return cv_precomputed(kmat(S, S), y)
    skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
    return np.array([classical_models()[model].fit(X[a], y[a]).score(X[b], y[b]) for a, b in skf.split(X, y)])

CV = {}
for case in CASES:
    for m in ["Lojistik R.", "RBF-SVM", "MLP (16)", "QSVM"]:
        sc = cv_scores(case, m); h = stats.t.ppf(0.975, len(sc) - 1) * sc.std(ddof=1) / np.sqrt(len(sc))
        CV[(case, m)] = (sc.mean(), sc.std(ddof=1), sc.mean() - h, min(sc.mean() + h, 1.0))
cv_df = pd.DataFrame([{"vaka": c, "model": m, "CV ort.": v[0], "std": v[1], "GA alt": v[2], "GA üst": v[3]} for (c, m), v in CV.items()])
cv_df.round(3)

**Yorum:** 3–8 vakasında dört modelin güven aralıkları **büyük ölçüde örtüşüyor**. Bu, "QSVM klasik modellerden daha iyidir" ya da "daha kötüdür" diyecek istatistiksel kanıt olmadığı anlamına gelir. Dürüst rapor bunu açıkça söyler.

---
## F · Maliyet analizi: devre sayısı × shot
Simülatörde çekirdek bir matris çarpımıdır; **gerçek donanımda** ise her K(x, x') değeri ayrı bir devre çalıştırılarak **shot** ile tahmin edilir.

| Bileşen | Devre sayısı |
|---|---|
| Eğitim çekirdeği (simetrik, köşegen = 1) | n(n − 1)/2 |
| Test çekirdeği | n_test × n |
| VQC, parameter-shift, epoch başına | n × (2·P + 1) |

In [ ]:
def kernel_cost(n_train, n_test, shots=1024):
    train = n_train * (n_train - 1) // 2
    test = n_test * n_train
    return train, test, (train + test) * shots

for case, (Xtr, Xte, ytr, yte) in SPLITS.items():
    tr, te, tot = kernel_cost(len(Xtr), len(Xte))
    print(f"{case:12s} eğitim devresi = {tr:6d}  test devresi = {te:6d}  toplam shot = {tot:,}")

# Gerçek transpile edilmiş çekirdek devresinin boyutu (6 kübit)
n = 6; fm6 = zz_feature_map(n, reps=2, entanglement="linear")
a, b = ParameterVector("a", n), ParameterVector("b", n)
kc6 = fm6.assign_parameters(a).compose(fm6.assign_parameters(b).inverse()); kc6.measure_all()
tq6 = transpile(kc6, basis_gates=["rz", "sx", "x", "cx"], optimization_level=1)
print("\n6 kübit çekirdek devresi:", dict(tq6.count_ops()), " derinlik =", tq6.depth())

---
## G · Gürültü deneyi: Aer gürültü modeliyle QSVM çekirdeği
7\. haftada gördüğümüz gürültü modellerini kullanıyoruz: tek kübit kapılarına p₁, CNOT'a p₂ olasılıklı **depolarize** hata, ölçüme p_r olasılıklı **okuma hatası**. Süreyi kısa tutmak için 3–8 vakasının **ilk 4 bileşeni**, 24 eğitim / 24 test örneği ve 256 shot kullanıyoruz.

Aer'de tek bir parametreli devreyi `parameter_binds` ile yüzlerce kez çalıştırmak, her devreyi ayrı ayrı oluşturmaktan çok daha hızlıdır.

In [ ]:
def noise_model(p1, p2, pr):
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(p1, 1), ["rz", "sx", "x"])
    nm.add_all_qubit_quantum_error(depolarizing_error(p2, 2), ["cx"])
    nm.add_all_qubit_readout_error(ReadoutError([[1 - pr, pr], [pr, 1 - pr]]))
    return nm

def shot_kernel(A, B, sim, shots=256, bw=BW):
    n = A.shape[1]; fm = zz_feature_map(n, reps=2, entanglement="linear")
    a, b = ParameterVector("a", n), ParameterVector("b", n)
    qc = fm.assign_parameters(a).compose(fm.assign_parameters(b).inverse()); qc.measure_all()
    tq = transpile(qc, basis_gates=["rz", "sx", "x", "cx"], optimization_level=1)
    pairs = [(i, j) for i in range(len(A)) for j in range(len(B))]
    binds = {p: [] for p in tq.parameters}
    for i, j in pairs:
        vals = {**{f"a[{k}]": A[i, k] * bw for k in range(n)}, **{f"b[{k}]": B[j, k] * bw for k in range(n)}}
        for p in tq.parameters: binds[p].append(vals[p.name])
    res = sim.run(tq, shots=shots, parameter_binds=[binds]).result()
    return np.array([res.get_counts(k).get("0" * n, 0) / shots for k in range(len(pairs))]).reshape(len(A), len(B))

df4 = CASES["3–8 (PCA 6)"][["pc1", "pc2", "pc3", "pc4", "y"]]
Ntr, Nte, nytr, nyte = train_test_split(df4.drop(columns="y").values, df4.y.values, train_size=24, test_size=24, stratify=df4.y.values, random_state=SEED)
Sa, Sb = fm_states(Ntr, BW), fm_states(Nte, BW); Kex = kmat(Sa, Sa)
NOISE = [{"ayar": "Kesin (Statevector)", "dogruluk": SVC(kernel="precomputed").fit(Kex, nytr).score(kmat(Sb, Sa), nyte), "K(x,x) ort.": 1.0}]
KN = {}
for lab, pr in [("İdeal + 256 shot", None), ("Düşük gürültü", (0.001, 0.01, 0.01)), ("Yüksek gürültü", (0.005, 0.05, 0.03))]:
    t = time.time()
    sim = AerSimulator(noise_model=noise_model(*pr), seed_simulator=7) if pr else AerSimulator(seed_simulator=7)
    Ktr_n = shot_kernel(Ntr, Ntr, sim); Kte_n = shot_kernel(Nte, Ntr, sim); KN[lab] = Ktr_n
    NOISE.append({"ayar": lab, "dogruluk": SVC(kernel="precomputed").fit(Ktr_n, nytr).score(Kte_n, nyte), "K(x,x) ort.": np.diag(Ktr_n).mean()})
    print(f"{lab:18s} bitti ({time.time()-t:.1f} s)")
noise_df = pd.DataFrame(NOISE); noise_df.round(3)

In [ ]:
iu = np.triu_indices(24, 1)
for lab, c in [("İdeal + 256 shot", GRAY), ("Düşük gürültü", BLUE), ("Yüksek gürültü", ORANGE)]:
    plt.scatter(Kex[iu], KN[lab][iu], s=8, color=c, label=lab)
plt.plot([0, 1], [0, 1], "k--", lw=0.8); plt.xlabel("kesin K"); plt.ylabel("ölçülen K"); plt.legend()
plt.title("Gürültü çekirdeği küçültür (depolarize → 1/2ⁿ'e doğru çeker)", color=NAVY); plt.show()

---
## H · Rapor tablosu
Tüm sonuçları tek bir `DataFrame`'de toplayıp **CSV** ve **markdown** olarak dışa aktarıyoruz. Dönem projesi raporunuzdaki sonuç tablosu bu şekilde üretilmelidir: elle kopyalanan sayı olmamalı.

In [ ]:
rep = pd.DataFrame(RESULTS)
rep["CV ort."] = [CV.get((r.vaka, r.model), (np.nan,))[0] for r in rep.itertuples()]
rep = rep.round(3)
rep.to_csv("hafta15_sonuclar.csv", index=False)
try:
    print(rep.to_markdown(index=False))
except ImportError:           # tabulate kurulu değilse
    print(rep.to_string(index=False))

pv = rep.pivot(index="model", columns="vaka", values="dogruluk").loc[["Lojistik R.", "RBF-SVM", "MLP (16)", "QSVM", "VQC"]]
pv.plot.bar(figsize=(9, 3.5), color=[BLUE, GRAY, ORANGE], ylim=(0.85, 1.01), rot=0)
plt.ylabel("test doğruluğu"); plt.title("Model başına test doğruluğu", color=NAVY); plt.legend(loc="lower left"); plt.show()

---
## I · Alıştırmalar
`# TODO` yerlerini doldurun; `assert` satırları geçerse çözüm doğrudur.

### Alıştırma 1 · Karışıklık matrisinden F1
`f1_from_cm(cm)` fonksiyonu sklearn düzenindeki `[[TN, FP], [FN, TP]]` matrisinden `(kesinlik, duyarlılık, F1)` döndürsün.

In [ ]:
def f1_from_cm(cm):
    # TODO
    pass

cm = np.array([[49, 6], [0, 53]])            # 3–8 VQC karışıklık matrisi
P, Rc, F = f1_from_cm(cm)
print(P, Rc, F)
assert np.isclose(P, 53/59) and np.isclose(Rc, 1.0)
assert np.isclose(F, 106/112)
yte = SPLITS["3–8 (PCA 6)"][3]; p = PREDS[("3–8 (PCA 6)", "QSVM")]
assert np.isclose(f1_from_cm(confusion_matrix(yte, p))[2], f1_score(yte, p))
print("Alıştırma 1 ✓")

### Alıştırma 2 · Çapraz doğrulama güven aralığı
`mean_ci(scores, conf=0.95)` → `(ortalama, alt, üst)` döndürsün. Örneklem standart sapması (`ddof=1`) ve t dağılımı (`stats.t.ppf`) kullanın.

In [ ]:
def mean_ci(scores, conf=0.95):
    # TODO
    pass

m, lo, hi = mean_ci([0.9583, 0.9861, 0.9718, 0.9859, 1.0])
print(round(m, 4), round(lo, 4), round(hi, 4))
assert np.isclose(m, 0.98042, atol=1e-4)
assert np.isclose(lo, 0.96072, atol=1e-3) and np.isclose(hi, 1.00012, atol=1e-3)
print("Alıştırma 2 ✓")

### Alıştırma 3 · Donanım maliyeti
`hardware_cost(n_train, n_test, shots, sec_per_shot)` → `(devre_sayısı, toplam_shot, saat)` döndürsün. Eğitim çekirdeği simetrik ve köşegeni 1 olduğu için n(n−1)/2 devre, test çekirdeği n_test·n_train devre gerektirir.

In [ ]:
def hardware_cost(n_train, n_test, shots=1024, sec_per_shot=1e-4):
    # TODO
    pass

c, s, h = hardware_cost(249, 108)
print(c, s, round(h, 2))
assert c == 249*248//2 + 108*249
assert s == c * 1024
assert np.isclose(h, s * 1e-4 / 3600)
print("Alıştırma 3 ✓")

### Alıştırma 4 · Kaç PCA bileşeni?
`n_components_for(a, b, target)` fonksiyonu, a–b rakam çifti için açıklanan varyansı en az `target` olan **en küçük** bileşen sayısını döndürsün (PCA'yı 64 bileşenle fit edip `explained_variance_ratio_` üzerinde kümülatif toplam alın).

In [ ]:
def n_components_for(a, b, target):
    # TODO
    pass

k01, k38 = n_components_for(0, 1, 0.70), n_components_for(3, 8, 0.70)
print(k01, k38)
assert k01 == 4 and k38 == 8
assert n_components_for(3, 8, 0.50) == 4
print("Alıştırma 4 ✓  -> 3 vs 8 aynı varyans için 2 kat daha fazla kübit istiyor")

### Alıştırma 5 · Çekirdek–hedef hizalaması
`alignment(K, y)` = ⟨K, Y⟩_F / (‖K‖_F · ‖Y‖_F), burada Y = y'y'ᵀ ve y' = ±1 etiketler. 3–8 eğitim kümesinin ilk 120 örneğinde `bw ∈ {0.05, 0.15, 1.0}` için hizalamayı hesaplayın. En iyi bant genişliği hangisi?

In [ ]:
def alignment(K, y):
    # TODO
    pass

Xtr, Xte, ytr, yte = SPLITS["3–8 (PCA 6)"]
Xa, ya = Xtr[:120], ytr[:120]
al = {bw: alignment(kmat(fm_states(Xa, bw), fm_states(Xa, bw)), ya) for bw in [0.05, 0.15, 1.0]}
print(al)
assert max(al, key=al.get) == 0.15
assert 0 < al[1.0] < al[0.15] < 1
print("Alıştırma 5 ✓")

### Alıştırma 6 · Çekirdek yoğunlaşmasını ölç
`offdiag_stats(X, bw)` → çekirdeğin köşegen dışı elemanlarının `(ortalama, varyans)` değerlerini döndürsün. 3–8 verisinin (PCA 8) ilk 60 örneğinde, bw = 1.0 için 2 ve 8 kübit (ilk 2 ve ilk 8 sütun) sonuçlarını karşılaştırın.

In [ ]:
def offdiag_stats(X, bw):
    # TODO
    pass

X8 = ds_digits_pair(3, 8, 8).drop(columns="y").values[:60]
m2, v2 = offdiag_stats(X8[:, :2], 1.0)
m8, v8 = offdiag_stats(X8[:, :8], 1.0)
print(f"2 kübit: ort {m2:.4f} var {v2:.5f}   8 kübit: ort {m8:.4f} var {v8:.6f}")
assert m8 < 0.05 < m2
assert v8 < v2 / 100
print("Alıştırma 6 ✓  -> 8 kübitte çekirdek neredeyse birim matris: yoğunlaşma")

### Alıştırma 7 · Öğrenme eğrisi (küçük veri)
3–8 vakasında eğitim kümesinden stratify ile `n ∈ {20, 80}` örnek seçip (random_state = 0) hem RBF-SVM hem QSVM (hazır `KERNELS` matrisinin alt blokları) test doğruluğunu hesaplayan `learning_point(n)` fonksiyonunu yazın. `(rbf_acc, qsvm_acc)` döndürsün.

In [ ]:
def learning_point(n):
    Xtr, Xte, ytr, yte = SPLITS["3–8 (PCA 6)"]; Ktr, Kte = KERNELS["3–8 (PCA 6)"]
    idx, _ = train_test_split(np.arange(len(ytr)), train_size=n, stratify=ytr, random_state=0)
    # TODO
    pass

r20, q20 = learning_point(20); r80, q80 = learning_point(80)
print(f"n=20: RBF {r20:.3f} QSVM {q20:.3f}   n=80: RBF {r80:.3f} QSVM {q80:.3f}")
assert q80 > q20 and r80 >= r20
assert q80 > 0.93 and r80 > 0.93
print("Alıştırma 7 ✓")

### Alıştırma 8 · Dürüst karar fonksiyonu
`compare(cv_a, cv_b)`: iki modelin CV skor listelerini alıp **güven aralıkları örtüşüyorsa** `"fark yok"`, değilse daha yüksek ortalamaya sahip modelin adını (`"A"` ya da `"B"`) döndürsün. Alıştırma 2'deki `mean_ci`'yi kullanın.

In [ ]:
def compare(cv_a, cv_b):
    # TODO
    pass

qsvm = cv_scores("3–8 (PCA 6)", "QSVM"); rbf = cv_scores("3–8 (PCA 6)", "RBF-SVM")
print("QSVM vs RBF-SVM:", compare(qsvm, rbf))
assert compare(qsvm, rbf) == "fark yok"
assert compare([0.70, 0.72, 0.71, 0.69, 0.70], [0.90, 0.91, 0.92, 0.90, 0.89]) == "B"
print("Alıştırma 8 ✓")

In [ ]:
print(f"Toplam çalışma süresi: {(time.time() - T_START)/60:.1f} dk")

---
### Haftanın (ve dersin) özeti
- Uçtan uca boru hattı: **problem → veri → ön işleme → kodlama → model → değerlendirme → klasik temel → maliyet → gürültü → rapor**
- Küçük ve kolay problemlerde (digits 0–1, 3–8) klasik modeller **en az kuantum modeller kadar iyi**, üstelik binlerce kat hızlı
- QSVM'de **bant genişliği** kritik: büyük ölçek → çekirdek yoğunlaşması → kötü genelleme
- Gürültü çekirdek değerlerini küçültür; donanımda çekirdek maliyeti **n² × shot** ile büyür
- Dürüst karşılaştırma: aynı ayrım, ayarlanmış klasik modeller, CV + güven aralığı, maliyet ve gürültü raporu

### Dönem projesi teslim takvimi (genel öneri)
| Adım | Zaman |
|---|---|
| Konu seçimi ve 1 sayfalık öneri | 1. hafta |
| Veri + klasik temel modeller | 2. hafta |
| Kuantum modeller ve deneyler | 3–4. hafta |
| Gürültü / maliyet analizi, rapor taslağı | 5. hafta |
| Rapor + notebook + sunum teslimi | 6. hafta |

Bu notebook'u kopyalayıp kendi veri setiniz ve modellerinizle doldurabilirsiniz. Başarılar!